
# UNet Fine-Tuning for Lake Segmentation — **RGB from Planet 8-Band**

This notebook trains a **UNet** on **RGB** channels extracted from **Planet 8-band** GeoTIFF tiles (size **512×512**).  
We use **ImageNet-pretrained** encoders (ResNet, EfficientNet, MobileNet, …) so transfer learning works as intended.

**What it does**
- Randomly sample **1,000** image/label pairs by matching filenames.
- Extract **RGB** from Planet 8-band tiles (defaults to **(6,4,2) = (Red, Green, Blue)** for SuperDove SR).
- Train/Val split (80/20).
- Train UNet across multiple **backbones** using **pretrained(ImageNet)** weights.
- **Loss:** BCE + Soft Dice (good for class imbalance).
- **Metrics:** IoU, Precision, Recall, F1, Accuracy on **train + val**, logged to CSV.
- Save **best-train-IoU** checkpoint per backbone.

> If your band ordering is different, **change `RGB_BANDS`** below.


## Dependencies

In [ ]:

# If needed, uncomment to install:
# %pip install -q torch torchvision timm segmentation-models-pytorch rasterio albumentations pandas scikit-image


## Configuration

In [1]:

from pathlib import Path
import random

# === REQUIRED: Paths ===
IMAGES_DIR = Path("D:/Thesis/glacial_lake_detection_thesis/Training/3 Preparing train data/rgb_outputs/subset_750/images")  # change to your imagery folder
LABELS_DIR = Path("D:/Thesis/glacial_lake_detection_thesis/Training/3 Preparing train data/rgb_outputs/subset_750/labels")  # change to your label folder (same filenames)

# === PlanetScope SuperDove SR default band mapping (1-indexed) ===
# 1: Coastal, 2: Blue, 3: Green I, 4: Green, 5: Yellow, 6: Red, 7: Red Edge, 8: NIR
# Use RGB = (Red, Green, Blue) => (6, 4, 2) by default.
#RGB_BANDS = (6, 4, 2)  # (R, G, B) 1-indexed
RGB_BANDS = (3, 2, 1)  # (R, G, B) 1-indexed

# === Outputs ===
OUTPUT_DIR = Path("./training_output_rgb/subset_750")
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)

# === Training config ===
NUM_SAMPLES = 750
RANDOM_SEED = 42
VAL_RATIO = 0.2
BATCH_SIZE = 2
NUM_WORKERS = 0
EPOCHS = 20
LR = 1e-4
MIN_LR = 1e-6

random.seed(RANDOM_SEED)


## Discover Pairs & Sample

In [2]:

def list_pairs(images_dir: Path, labels_dir: Path):
    img_files = {p.name: p for p in images_dir.glob("*.tif")}
    lbl_files = {p.name: p for p in labels_dir.glob("*.tif")}
    common = sorted(set(img_files) & set(lbl_files))
    return [(img_files[n], lbl_files[n]) for n in common]

all_pairs = list_pairs(IMAGES_DIR, LABELS_DIR)
if not all_pairs:
    raise RuntimeError(f"No paired .tif files found under {IMAGES_DIR} & {LABELS_DIR}.")

print(f"Found {len(all_pairs)} pairs.")
random.shuffle(all_pairs)
sampled_pairs = all_pairs[:min(NUM_SAMPLES, len(all_pairs))]

split_idx = int(len(sampled_pairs) * (1 - VAL_RATIO))
train_pairs = sampled_pairs[:split_idx]
val_pairs = sampled_pairs[split_idx:]
print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")


Found 750 pairs.
Train: 600, Val: 150



## Dataset & Dataloaders (RGB extraction)
- Reads 8-band imagery and **selects RGB channels** via `RGB_BANDS` (1-indexed).
- Per-image **min–max** normalization to [0,1].
- Light flips for augmentation.


In [3]:

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
import albumentations as A

def read_geotiff(path: Path):
    with rasterio.open(path) as src:
        arr = src.read()  # (C,H,W), 1-indexed bands in file
    return arr

def select_rgb(x: np.ndarray, rgb_bands=(6,4,2)):
    idx = [b-1 for b in rgb_bands]
    return x[idx, ...].astype(np.float32)

def minmax_per_band(x: np.ndarray, eps=1e-6):
    x = x.astype(np.float32)
    for i in range(x.shape[0]):
        vmin = np.min(x[i]); vmax = np.max(x[i])
        if vmax - vmin < eps:
            x[i] = 0.0
        else:
            x[i] = (x[i] - vmin) / (vmax - vmin + eps)
    return x

class LakeTilesRGB(Dataset):
    def __init__(self, pairs, rgb_bands=(6,4,2), aug=None, normalize=True):
        self.pairs = pairs
        self.rgb_bands = rgb_bands
        self.aug = aug
        self.normalize = normalize

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        img8 = read_geotiff(img_path)          # (8,H,W)
        img = select_rgb(img8, self.rgb_bands) # (3,H,W)

        with rasterio.open(lbl_path) as src:
            lbl = src.read(1)                  # (H,W)
        lbl = (lbl > 0).astype(np.float32)

        if self.normalize:
            img = minmax_per_band(img)

        # Albumentations expects HWC
        img_hwc = np.transpose(img, (1,2,0))
        lbl_hwc = lbl[..., None]

        if self.aug:
            transformed = self.aug(image=img_hwc, mask=lbl_hwc)
            img_hwc = transformed["image"]
            lbl_hwc = transformed["mask"]

        # Back to CHW
        img = torch.from_numpy(np.transpose(img_hwc, (2,0,1))).float()
        lbl = torch.from_numpy(lbl_hwc[...,0]).float()
        return img, lbl

train_aug = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5)])
val_aug   = A.Compose([])

train_ds = LakeTilesRGB(train_pairs, rgb_bands=RGB_BANDS, aug=None, normalize=True)
val_ds   = LakeTilesRGB(val_pairs,   rgb_bands=RGB_BANDS, aug=None,   normalize=True)

PIN_MEMORY = False
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          pin_memory=PIN_MEMORY)

len(train_ds), len(val_ds)


(600, 150)

## Loss & Metrics

In [4]:

import torch.nn.functional as F

def dice_loss_with_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    if probs.dim() == 4:
        probs = probs[:,0]
    intersection = (probs * targets).sum(dim=(1,2))
    union = probs.sum(dim=(1,2)) + targets.sum(dim=(1,2)) + eps
    dice = (2 * intersection + eps) / union
    return 1 - dice.mean()

def bce_dice_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets) + dice_loss_with_logits(logits, targets)

def compute_metrics_from_logits(logits, targets):
    with torch.no_grad():
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        t = targets.float()
        tp = (preds * t).sum(dim=(1,2))
        tn = ((1-preds) * (1-t)).sum(dim=(1,2))
        fp = (preds * (1-t)).sum(dim=(1,2))
        fn = ((1-preds) * t).sum(dim=(1,2))
        eps = 1e-7
        precision = (tp / (tp + fp + eps)).mean().item()
        recall    = (tp / (tp + fn + eps)).mean().item()
        f1        = (2*precision*recall / (precision + recall + eps))
        iou       = (tp / (tp + fp + fn + eps)).mean().item()
        accuracy  = ((tp + tn) / (tp + tn + fp + fn + eps)).mean().item()
        return dict(iou=iou, precision=precision, recall=recall, f1=f1, accuracy=accuracy)


## Model & Training (ImageNet-pretrained encoders, in_channels=3)

In [5]:

import segmentation_models_pytorch as smp
import torch
from torch import optim
import pandas as pd
from datetime import datetime
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


def _sinusoidal_position_encoding_2d(h, w, d_model, device):
    """
    2D sinusoidal PE → (h*w, d_model), interleaving sin/cos for Y and X.
    Works for any H×W, no learned params.
    """
    pe = torch.zeros(d_model, h, w, device=device)

    # ensure multiple of 4 so we can split evenly into (y_sin, y_cos, x_sin, x_cos)
    base = (d_model // 4) * 4
    if base == 0:
        raise ValueError("d_model must be >= 4")

    # number of channels per each of the four groups
    dim_each = base // 4  # e.g., 512 -> 128

    # frequency terms for Y and X (same length)
    div_term = torch.exp(torch.arange(dim_each, device=device) * (-math.log(10000.0) / dim_each))

    # positions
    pos_y = torch.arange(h, device=device).unsqueeze(1)     # (H,1)
    pos_x = torch.arange(w, device=device).unsqueeze(1)     # (W,1)

    # ----- Y encodings -----
    # shapes after ops: (dim_each, H, W)
    y_sin = torch.sin(pos_y * div_term).transpose(0, 1).unsqueeze(2).repeat(1, 1, w)
    y_cos = torch.cos(pos_y * div_term).transpose(0, 1).unsqueeze(2).repeat(1, 1, w)

    # place into interleaved channels [0,2,4,...] and [1,3,5,...] up to base//2
    pe[0:base//2:2, :, :] = y_sin
    pe[1:base//2:2, :, :] = y_cos

    # ----- X encodings -----
    # shapes after ops: (dim_each, H, W)
    x_sin = torch.sin(pos_x * div_term).transpose(0, 1).unsqueeze(1).repeat(1, h, 1)
    x_cos = torch.cos(pos_x * div_term).transpose(0, 1).unsqueeze(1).repeat(1, h, 1)

    # place into interleaved channels in the second half
    pe[base//2::2, :, :]     = x_sin
    pe[base//2+1::2, :, :]   = x_cos

    # if d_model wasn't divisible by 4, pad remaining channels with zeros
    if base < d_model:
        pe = F.pad(pe, (0, 0, 0, 0, 0, d_model - base))

    # return tokens x dim
    pe = pe.view(d_model, h * w).transpose(0, 1)  # (h*w, d_model)
    return pe


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        # align shapes in case of off-by-one
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class TransformerBottleneck(nn.Module):
    def __init__(self, d_model=512, n_heads=8, depth=4, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=int(d_model*mlp_ratio),
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
        self.d_model = d_model

    def forward(self, x):
        # x: (B, C, H, W) with C == d_model
        B, C, H, W = x.shape
        tokens = x.flatten(2).transpose(1,2)  # (B, H*W, C)
        pe = _sinusoidal_position_encoding_2d(H, W, C, x.device).unsqueeze(0).repeat(B,1,1)
        tokens = tokens + pe
        tokens = self.encoder(tokens)  # (B, H*W, C)
        out = tokens.transpose(1,2).view(B, C, H, W)
        return out

class TransUNet(nn.Module):
    """
    ResNet18 encoder (adjusted to in_channels), ViT bottleneck on the last feature map,
    and UNet-style decoder with skip connections from ResNet stages.
    """
    def __init__(self, in_channels=3, num_classes=1, transformer_dim=512, heads=8, depth=4, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        resnet = models.resnet18(weights=None)  # no internet; do not fetch weights
        # Patch the first conv for custom input channels
        if in_channels != 3:
            conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
            resnet.conv1 = conv1
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.enc1 = resnet.layer1  # 64
        self.enc2 = resnet.layer2  # 128
        self.enc3 = resnet.layer3  # 256
        self.enc4 = resnet.layer4  # 512

        self.to_transformer = nn.Conv2d(512, transformer_dim, kernel_size=1, bias=False)
        self.transformer = TransformerBottleneck(
            d_model=transformer_dim, n_heads=heads, depth=depth, mlp_ratio=mlp_ratio, dropout=dropout
        )
        self.from_transformer = nn.Conv2d(transformer_dim, 512, kernel_size=1, bias=False)

        # Decoder
        self.up3 = UpBlock(512, 256, 256)
        self.up2 = UpBlock(256, 128, 128)
        self.up1 = UpBlock(128, 64, 64)
        self.up0 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec0 = ConvBlock(32, 32)

        self.head = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        x0 = self.stem(x)        # 64, /2
        x0p = self.pool(x0)      # /4
        x1 = self.enc1(x0p)      # 64, /4
        x2 = self.enc2(x1)       # 128, /8
        x3 = self.enc3(x2)       # 256, /16
        x4 = self.enc4(x3)       # 512, /32

        # Transformer on bottleneck
        t = self.to_transformer(x4)
        t = self.transformer(t)
        x4t = self.from_transformer(t)

        # Decoder with skips
        d3 = self.up3(x4t, x3)   # -> 256, /16
        d2 = self.up2(d3, x2)    # -> 128, /8
        d1 = self.up1(d2, x1)    # -> 64, /4
        d0 = self.up0(d1)        # -> 32, /2
        d0 = self.dec0(d0)

        # Final upsample to input size if needed
        logit = self.head(d0)
        if logit.shape[-2:] != x.shape[-2:]:
            logit = F.interpolate(logit, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return logit


def make_transunet_rgb(num_classes: int = 1):
    return TransUNet(in_channels=3, num_classes=num_classes,
                         transformer_dim=512, heads=8, depth=6, mlp_ratio=4.0, dropout=0.0)

def run_one_epoch(model, loader, optimizer=None, use_amp=True):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0; n_batches = 0
    agg = dict(iou=0.0, precision=0.0, recall=0.0, f1=0.0, accuracy=0.0)

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        if is_train: optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)[:,0]
            loss = bce_dice_loss(logits, lbls)

        if is_train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item(); n_batches += 1
        m = compute_metrics_from_logits(logits.detach(), lbls.detach())
        for k in agg: agg[k] += m[k]

    avg_loss = total_loss / max(1, n_batches)
    for k in agg: agg[k] /= max(1, n_batches)
    torch.cuda.empty_cache()
    return avg_loss, agg

# Determinism tweaks (optional)
#torch.backends.cudnn.benchmark = False
#torch.backends.cudnn.deterministic = True


Device: cuda


In [6]:
# === Optuna hyperparameter search for TransUNet bottleneck ===
# pip install optuna   # <- do this once in your env
import optuna, time, torch

def objective(trial):
    heads      = trial.suggest_categorical("heads", [4, 8])
    depth      = trial.suggest_categorical("depth", [2, 4, 6])
    mlp_ratio  = trial.suggest_float("mlp_ratio", 2.0, 5.0, step=1.0)  # {2,3,4,5,6}

    model = TransUNet(in_channels=3, num_classes=1,
                      transformer_dim=512,
                      heads=heads, depth=depth, mlp_ratio=mlp_ratio,
                      dropout=0.0).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    max_epochs = 5  # short runs for search

    best_val_iou = -1.0
    for epoch in range(1, max_epochs+1):
        train_loss, _ = run_one_epoch(model, train_loader, optimizer, use_amp=True)
        val_loss,   val_metrics = run_one_epoch(model, val_loader, optimizer=None, use_amp=True)
        best_val_iou = max(best_val_iou, float(val_metrics.get("iou", 0.0)))

        # Prune bad trials early
        trial.report(best_val_iou, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    # We want to maximize IoU (Optuna minimizes by default if you set direction="minimize")
    return best_val_iou

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, timeout=None)  # adjust trials/budget

print("Best value (IoU):", study.best_value)
print("Best params:", study.best_params)

# Use the best params in your main training cell:
best = study.best_params
print(f"\n--> Set in TransUNet(): heads={best['heads']}, depth={best['depth']}, mlp_ratio={best['mlp_ratio']}")


[I 2025-11-02 18:03:01,995] A new study created in memory with name: no-name-84224c39-2877-4888-8465-0614b9b9b4c5
c:\Users\gg\anaconda3\envs\thesis\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
C:\Users\gg\AppData\Local\Temp\ipykernel_5628\4103995556.py:186: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\gg\AppData\Local\Temp\ipykernel_5628\4103995556.py:197: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
[I 2025-11-02 18:05:32,418] Trial 0 finished with value: 0.5205038820207118 and parameters: {'heads': 4, 'depth': 4, 'mlp_ratio': 5.0}. Best is trial 0 with value: 0.5205038820

Best value (IoU): 0.5397251397371292
Best params: {'heads': 4, 'depth': 6, 'mlp_ratio': 3.0}

--> Set in TransUNet(): heads=4, depth=6, mlp_ratio=3.0


In [8]:
from torch.optim.lr_scheduler import CosineAnnealingLR

results_csv = OUTPUT_DIR / "logs" / "results.csv"
if not results_csv.exists():
    pd.DataFrame(columns=[
        "timestamp","epoch","split",
        "loss","iou","precision","recall","f1","accuracy"
    ]).to_csv(results_csv, index=False)

model = TransUNet(in_channels=3, num_classes=1,
                      transformer_dim=512,
                      heads=8, depth=6, mlp_ratio=5,
                      dropout=0.0).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
#scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)

best_val_iou = -1.0
best_path = OUTPUT_DIR / "models" / f"best_val_iou.pt"

for epoch in range(1, EPOCHS+1):
    train_loss, train_metrics = run_one_epoch(model, train_loader, optimizer, use_amp=True)
    val_loss,   val_metrics   = run_one_epoch(model, val_loader, optimizer=None, use_amp=True)
    #scheduler.step()

    print(f"Epoch {epoch}/{EPOCHS} — "
            f"train IoU={train_metrics['iou']:.4f}  val IoU={val_metrics['iou']:.4f}  "
            f"train Loss={train_loss:.4f}  val Loss={val_loss:.4f}")

    ts = datetime.utcnow().isoformat()
    pd.DataFrame([
        dict(timestamp=ts, epoch=epoch, split="train", loss=train_loss, **train_metrics),
        dict(timestamp=ts, epoch=epoch, split="val",   loss=val_loss,   **val_metrics),
    ]).to_csv(results_csv, mode="a", index=False, header=False)

    if val_metrics["iou"] > best_val_iou:
        best_val_iou = val_metrics["iou"]
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_iou": best_val_iou,
        }, best_path)

print(f"Training complete. Logs: {results_csv}")
print(f"Models saved under: {OUTPUT_DIR / 'models'}")


C:\Users\gg\AppData\Local\Temp\ipykernel_5628\4103995556.py:186: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\gg\AppData\Local\Temp\ipykernel_5628\4103995556.py:197: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 1/20 — train IoU=0.2379  val IoU=0.3781  train Loss=1.3682  val Loss=1.2260


C:\Users\gg\AppData\Local\Temp\ipykernel_5628\347504635.py:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat()


Epoch 2/20 — train IoU=0.3305  val IoU=0.3578  train Loss=1.2102  val Loss=1.1450
Epoch 3/20 — train IoU=0.3722  val IoU=0.4505  train Loss=1.1115  val Loss=1.0284
Epoch 4/20 — train IoU=0.3836  val IoU=0.4808  train Loss=1.0333  val Loss=0.9429
Epoch 5/20 — train IoU=0.4205  val IoU=0.4487  train Loss=0.9530  val Loss=0.9036
Epoch 6/20 — train IoU=0.4405  val IoU=0.5308  train Loss=0.8835  val Loss=0.8187
Epoch 7/20 — train IoU=0.4540  val IoU=0.5115  train Loss=0.8131  val Loss=0.7431
Epoch 8/20 — train IoU=0.4773  val IoU=0.4907  train Loss=0.7431  val Loss=0.7098
Epoch 9/20 — train IoU=0.4852  val IoU=0.5418  train Loss=0.6821  val Loss=0.6204
Epoch 10/20 — train IoU=0.5121  val IoU=0.5736  train Loss=0.6176  val Loss=0.5710
Epoch 11/20 — train IoU=0.5204  val IoU=0.5987  train Loss=0.5704  val Loss=0.4924
Epoch 12/20 — train IoU=0.5220  val IoU=0.5699  train Loss=0.5387  val Loss=0.5077
Epoch 13/20 — train IoU=0.5397  val IoU=0.5821  train Loss=0.4958  val Loss=0.4502
Epoch 14/20 